# Create World Food Prize Awards (PRIZE PATTERN, decade-bucket scrape)

Ingest of the **World Food Prize**, awarded annually since 1987 by the **World Food Prize Foundation** for "outstanding contributions to improving the quality, quantity, or availability of food in the world." The prize was founded by Norman Borlaug (the 1970 Nobel Peace Prize laureate). Notable laureates: M.S. Swaminathan (inaugural 1987), Muhammad Yunus (1994), Pedro Sanchez (2002), Senators Bob Dole & George McGovern (joint 2008), Cary Fowler & Geoffrey Hawtin (joint 2024, for the Svalbard Global Seed Vault).

**Awarding body:** World Food Prize Foundation — `F4320308859` (US, DOI `10.13039/100005989`, no ROR). Alternate titles include `WFPF`, `The World Food Prize`, plus localized variants (Spanish, French, Arabic, Russian, Chinese).

**Source:** `worldfoodprize.org/en/laureates/` — the foundation's own site (Webflow + sitevizcms CMS, first ingest on the project to scrape this CMS family). Laureates are organized into **4 decade-bucketed listing pages**:
- `/en/laureates/19871999_laureates/` — 12 detail pages (1990 has a duplicate URL `niderhauser` vs `niederhauser`; the script dedupes)
- `/en/laureates/20002009_laureates/` — 10
- `/en/laureates/20102019_laureates/` — 10
- `/en/laureates/20202026_laureates/` — 7

Each decade page links to per-laureate detail pages at `/en/laureates/{decade}_laureates/{year}_{slug}/` where the slug is the surname(s) — joint awards use `surname_and_surname` (1996 Beachell+Khush) or `surname_surname_surname` (2006 Lobato+McClung+Paolinelli; 2013 van Montagu+Chilton+Fraley).

Method #5 on the runbook ladder (static-HTML scrape), specifically the **"decade-bucket" variant** — corpus split into 4 listing pages, each linked to N detail pages. This contrasts with Stockholm Water Prize's "big-page-inline" pattern (everything in one page) and Templeton's "single listing page" pattern (all cards on one URL with no per-laureate fetch needed).

**Corpus** (verified 2026-05-22 full local scrape): **39 laureate rows** across **1987-2026** (40 years, with **1997 missing** — the foundation links to `/1997_adkisson_and_smith/` but that detail page returns 404 from their CMS, a real foundation-side broken link, not a parser bug).
- 27 single-laureate years + 12 joint years (1992/1996/1997/2000/2004/2006/2008/2010/2011/2013/2016/2018/2024)
- 82% given_name coverage (32/39) — the 7 misses are joint-award detail pages whose `<h2>` displays a combined name (e.g., "Beachell and Khush") without a clear first-recipient given-name
- 62% description coverage (24/39) — older detail pages (pre-2010) don't always have a `Full Biography` section; we capture what's present

**Schema:**
- `funder_award_id` = `world-food-prize-{year}-{slug}` (e.g., `world-food-prize-1987-swaminathan`). Verified unique.
- `lead_investigator` is PERSON-LEVEL: `given_name` + `family_name` parsed from the page's `<h2>` full-name heading (runbook §2.4.1 split). For joint awards, only the FIRST recipient becomes `lead_investigator`; the co-laureate(s) are captured in `description`.
- `affiliation` is NULL — the foundation's detail pages narrate biographies but don't expose an institutional affiliation field. Could be enriched in a follow-up.
- `funder_scheme = 'World Food Prize'`. Single program.
- `funding_type = 'prize'`.
- `declined` always FALSE.

**Amount choice — constant USD 500,000 per laureate:**
The foundation's own `/en/laureates/` index page states verbatim: **"The Laureate receives a cash prize of $500,000, recognizing their outstanding contributions to improving global food security."** (Verified 2026-05-22.) We ship USD 500,000 uniformly across all years. `currency='USD'`. **§6.7 amount-coverage NOT waived** — 100% expected.

**Source authority** — `worldfoodprize.org` is the foundation's own site. Webflow/sitevizcms CMS, server-side-rendered detail pages. Method #5 (static-HTML scrape) on the runbook ladder, decade-bucket variant.

**Prerequisites**: Run `scripts/local/world_food_prize_to_s3.py` first to fetch the 4 decade pages + 40 detail pages (~25 sec wall-clock at 0.4s throttle, ~39 successful records after one broken link drops 1997), write parquet, upload to S3.

**Data source**: `https://www.worldfoodprize.org/en/laureates/` (index) + 4 decade-bucket sub-pages

**S3 location**: `s3a://openalex-ingest/awards/world_food_prize/world_food_prize_laureates.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.world_food_prize_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/world_food_prize/world_food_prize_laureates.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.world_food_prize_raw;

## Step 1.5: Inspect raw + money/currency scan + coverage acknowledgment

All source columns from the decade-bucket scrape. **Verified per-row coverage on the full extraction** (39 laureate rows, validated 2026-05-22):
- 100% `funder_award_id`, `year`, `slug`, `name`, `landing_page_url`
- 100% `amount` = constant $500,000 USD per laureate (foundation's documented rule)
- 82% `given_name` (32/39 — joint-award entries whose `<h2>` shows a combined name carry only `family_name`)
- 100% `family_name`
- 62% `description` (24/39 — older detail pages pre-2010 sometimes lack `Full Biography` section)
- 39 distinct years 1987-2026 (with **1997 missing** due to foundation-side broken link — they removed the Adkisson+Smith detail page; documented in tracker)
- 12 joint-laureate years

Source columns: `funder_award_id`, `year`, `slug`, `name`, `given_name`, `family_name`, `is_joint`, `h1_surname_block`, `display_name`, `description`, `amount`, `currency`, `start_date`, `end_date`, `landing_page_url`, `declined`.

In [ ]:
%sql
DESCRIBE openalex.awards.world_food_prize_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.world_food_prize_raw LIMIT 5;

In [ ]:
%sql
-- §1.5 money-shape scan — amount is the foundation's documented constant
-- $500K USD ceiling.
SELECT
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(amount AS DOUBLE)) AS avg_amount,
    COUNT(amount) AS non_null,
    COUNT(*) AS total_rows
FROM openalex.awards.world_food_prize_raw;

In [ ]:
%sql
-- Year + joint-laureate distribution
SELECT TRY_CAST(year AS INT) AS year_int,
       is_joint,
       COUNT(*) AS rows
FROM openalex.awards.world_food_prize_raw
GROUP BY year_int, is_joint
ORDER BY year_int;

## Step 1.6: Fail-fast — verify World Food Prize Foundation funder row exists

In [ ]:
%sql
-- Must return exactly 1 row.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320308859;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.world_food_prize_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320308859  -- World Food Prize Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    n.display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'prize' AS funding_type,
    'World Food Prize' AS funder_scheme,
    'world_food_prize' AS provenance,
    TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_date,   'yyyy-MM-dd') AS end_date,
    TRY_CAST(SUBSTRING(n.start_date, 1, 4) AS INT) AS start_year,
    TRY_CAST(SUBSTRING(n.end_date,   1, 4) AS INT) AS end_year,
    -- lead_investigator: PERSON-LEVEL. For joint awards, captures the first
    -- recipient only (the co-laureate appears in the description). Affiliation
    -- is NULL — the foundation's pages narrate biographies but don't expose
    -- an institutional field.
    CASE
        WHEN n.name IS NULL OR n.name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
            struct(
                CAST(NULL AS STRING) AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    n.declined,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.world_food_prize_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 101

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'world_food_prize' AND priority = 101;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    101 AS priority  -- World Food Prize priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.world_food_prize_awards;

## Step 6: Verification

Full §6.1-6.7. Amount-coverage NOT waived (constant $500K per laureate). Verified expectations from the 2026-05-22 extraction:
- Row count: **39** (full roster minus 1997 Adkisson+Smith — foundation-side 404 on their detail page)
- 100% `pct_amount` (constant $500,000 USD per laureate)
- `currencies = [USD]`
- 1 distinct `funder_scheme` ('World Food Prize'), 1 distinct `funding_type` ('prize')
- 39 distinct years (1987-2026, gap at 1997)
- 12 joint-laureate years
- `declined = FALSE` everywhere

In [ ]:
%sql
SELECT COUNT(*) AS total_wfp_award_rows FROM openalex.awards.world_food_prize_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.world_food_prize_awards;

In [ ]:
%sql
-- §6.3 Data completeness
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description
FROM openalex.awards.world_food_prize_awards;

In [ ]:
%sql
-- §6.7 amount + currency fail-fast (NOT waived).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount
FROM openalex.awards.world_food_prize_awards;

In [ ]:
%sql
-- Year distribution + total prize dollars over time
SELECT start_year,
       COUNT(*) AS rows,
       ROUND(SUM(amount)/1e6, 2) AS total_usd_millions
FROM openalex.awards.world_food_prize_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year ASC
LIMIT 100;

In [ ]:
%sql
-- Sample notable + recent laureates
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       amount, start_year
FROM openalex.awards.world_food_prize_awards
WHERE start_year >= 2022 OR start_year <= 1990
ORDER BY start_year DESC
LIMIT 12;

In [ ]:
%sql
-- Joint-award rows (should be 12)
SELECT start_year, SUBSTRING(display_name, 1, 80) AS title
FROM openalex.awards.world_food_prize_awards
WHERE display_name LIKE '%and%' OR display_name LIKE '%&%'
ORDER BY start_year;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.world_food_prize_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Declined must be FALSE.
SELECT declined, COUNT(*) AS rows
FROM openalex.awards.world_food_prize_awards
GROUP BY declined;